In [1]:
#| default_exp diffusion_utils
import sys
sys.path.append("..")

In [2]:
#| export
import numpy as np
import torch 
import torch.nn as nn
from ddpm_backtest.models import FiLMLayer,f_net
from ddpm_backtest.noising_time import cosine_beta_scheduler,timestep_embedding

In [3]:
#| export
class TabularDDPM(nn.Module):
    def __init__(self, d_in, model, cond_in_classes, continious_conditioning_in, t_dim=128):
        super().__init__()
        self.cond_emb = nn.Embedding(cond_in_classes + 1, t_dim, padding_idx=0)
        self.cond_continious = nn.Linear(continious_conditioning_in, t_dim)
        self.net = model(d_in + t_dim + t_dim + t_dim, d_in)

    def forward(self, x, c, cond, t):
        t_emb = timestep_embedding(t, dim=128)
        c_emb = self.cond_emb(c).to(x.device)      
        c_cont = self.cond_continious(cond).to(x.device) 
        x_in = torch.cat([x, c_emb, c_cont, t_emb], dim=1)
        return self.net(x_in)

In [ ]:
#| export
class FiLMTabularDDPM(nn.Module):
    def __init__(self, d_in, model, cond_in_classes, continious_conditioning_in, t_dim=128):
        super().__init__()
        self.t_dim = t_dim
        self.cond_emb = nn.Embedding(cond_in_classes + 1, t_dim, padding_idx=0)
        self.cond_continious = nn.Sequential(
            nn.Linear(continious_conditioning_in, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim), 
        )
        hidden = 256
        self.fc1    = nn.Linear(d_in + t_dim + t_dim, hidden)  
        self.fc2    = nn.Linear(hidden, 512)
        self.fc3    = nn.Linear(512, 512)
        self.fc4    = nn.Linear(512, 256)
        self.fc5    = nn.Linear(256, hidden)
        self.fc_out = nn.Linear(hidden, d_in)
        self.act    = nn.SiLU()
        self.film1  = FiLMLayer(hidden, t_dim)
        self.film2  = FiLMLayer(512,    t_dim)
        self.film3  = FiLMLayer(512,    t_dim)
        self.film4  = FiLMLayer(256,    t_dim)
        self.film5  = FiLMLayer(hidden, t_dim)

    def forward(self, x, c, cond, t):
        t_emb  = timestep_embedding(t, dim=self.t_dim).to(x.device)
        c_emb  = self.cond_emb(c).to(x.device)
        c_cont = self.cond_continious(cond).to(x.device)
        h = torch.cat([x, c_emb, t_emb], dim=1)

        h = self.act(self.fc1(h)); h = self.film1(h, c_cont)
        h = self.act(self.fc2(h)); h = self.film2(h, c_cont)
        h = self.act(self.fc3(h)); h = self.film3(h, c_cont)
        h = self.act(self.fc4(h)); h = self.film4(h, c_cont)
        h = self.act(self.fc5(h)); h = self.film5(h, c_cont)

        return self.fc_out(h)
